In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/q10_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

# Optimized for cudf: avoid CPU-side Timestamp conversions
osel = (orders.O_ORDERDATE >= '1994-11-01') & (orders.O_ORDERDATE < '1995-02-01')
lsel = lineitem.L_RETURNFLAG == 'R'

# Filter once
forders = orders[osel]
flineitem = lineitem[lsel]

# Chain merges to minimize intermediates
jn3 = (flineitem
       .merge(forders,     left_on='L_ORDERKEY', right_on='O_ORDERKEY')
       .merge(customer,    left_on='O_CUSTKEY',  right_on='C_CUSTKEY')
       .merge(nation,      left_on='C_NATIONKEY',right_on='N_NATIONKEY'))

# Compute TMP and aggregate
jn3['TMP'] = jn3.L_EXTENDEDPRICE * (1.0 - jn3.L_DISCOUNT)
total = (
    jn3
    .groupby(
        ['C_CUSTKEY','C_NAME','C_ACCTBAL','C_PHONE','N_NAME','C_ADDRESS','C_COMMENT'],
        as_index=False,
        sort=False
    )['TMP']
    .sum()
    .sort_values('TMP', ascending=False)
)
